# Maritime Port Logistics — Intent Classification Analysis
## PEFT/LoRA Fine-Tuning of RoBERTa-base

This notebook provides an in-depth analysis of a multi-class intent classification system
built for the maritime and port logistics domain. We fine-tuned `roberta-base` using
**Parameter-Efficient Fine-Tuning (PEFT)** with **Low-Rank Adaptation (LoRA)**, training
less than 1% of the model's parameters while achieving **93% accuracy** and **0.928 macro F1**.

**Key questions this notebook answers:**
1. How is the dataset structured and distributed?
2. How does LoRA work and what did we configure?
3. How does the fine-tuned model compare to the bare base model?
4. Where does the model struggle, and why?
5. How confident is the model in its predictions?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)

print("Setup complete.")

## 1. Dataset Overview

We use a **balanced training set** (~1,458 examples) paired with an **imbalanced test set**
(~400 examples) that reflects real-world intent frequencies. Class weights during training
compensate for the imbalance the model will face in production.

In [ ]:
# Load datasets
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")
label_map = pd.read_csv("../data/label_map.csv")

with open("../data/class_weights.json") as f:
    class_weights = json.load(f)

id2label = dict(zip(label_map["label"], label_map["intent_name"]))

print(f"Training set:  {len(train_df):,} examples")
print(f"Test set:      {len(test_df):,} examples")
print(f"Intent classes: {len(id2label)}")
print()
label_map

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Training distribution
train_counts = train_df["intent_name"].value_counts()
train_counts.sort_index().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Training Set Distribution (Balanced)")
axes[0].set_xlabel("Count")

# Test distribution
test_counts = test_df["intent_name"].value_counts()
test_counts.sort_index().plot.barh(ax=axes[1], color="coral")
axes[1].set_title("Test Set Distribution (Real-World Imbalanced)")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

print("\nTest set intent distribution:")
for intent in sorted(test_counts.index):
    count = test_counts[intent]
    pct = count / len(test_df) * 100
    print(f"  {intent:30s}  {count:4d}  ({pct:5.1f}%)")

## 2. Class Weights Strategy

Since rare intents like `report_port_incident` appear infrequently in production,
we apply **inverse-frequency class weights** during training. This makes the model
penalize misclassifications of rare intents more heavily, preventing the model from
simply ignoring them in favor of majority classes.

In [ ]:
# Visualize class weights
weight_df = pd.DataFrame([
    {"intent": id2label[int(k)], "weight": v}
    for k, v in class_weights.items()
]).sort_values("weight", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(weight_df["intent"], weight_df["weight"], color="teal")
ax.axvline(x=1.0, color="red", linestyle="--", alpha=0.7, label="Neutral weight (1.0)")
ax.set_xlabel("Class Weight")
ax.set_title("Training Class Weights (Inverse Real-World Frequency)")
ax.legend()

for bar, w in zip(bars, weight_df["weight"]):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f"{w:.1f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

print("Interpretation:")
print("  weight > 1.0 → rare intent, errors penalized more")
print("  weight < 1.0 → common intent, errors penalized less")
print("  weight = 1.0 → neutral")

In [ ]:
train_df["text_length"] = train_df["text"].str.len()
train_df["word_count"] = train_df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character length by intent
train_df.boxplot(column="text_length", by="intent_name", ax=axes[0], vert=False)
axes[0].set_title("Character Length by Intent")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("")

# Word count by intent
train_df.boxplot(column="word_count", by="intent_name", ax=axes[1], vert=False)
axes[1].set_title("Word Count by Intent")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("")

plt.suptitle("")
plt.tight_layout()
plt.show()

print(f"Overall text length — median: {train_df['text_length'].median():.0f} chars, "
      f"mean: {train_df['text_length'].mean():.1f} chars")
print(f"Overall word count  — median: {train_df['word_count'].median():.0f} words, "
      f"mean: {train_df['word_count'].mean():.1f} words")

## 3. LoRA Architecture & Configuration

**Low-Rank Adaptation (LoRA)** freezes the pre-trained model weights and injects
small trainable rank-decomposition matrices into the transformer's attention layers.

For a weight matrix **W** ∈ ℝ^(d×k), LoRA approximates the update as:

> **W' = W + ΔW = W + B·A**

where **B** ∈ ℝ^(d×r) and **A** ∈ ℝ^(r×k), with rank **r << min(d, k)**.

### Our Configuration

| Parameter | Value | Rationale |
|---|---|---|
| `r` (rank) | 8 | Good balance of capacity vs. efficiency |
| `alpha` | 16 | Scaling factor = alpha/r = 2× |
| `dropout` | 0.1 | Regularization for small datasets |
| `target_modules` | query, value | Standard attention projections |
| `modules_to_save` | classifier | Classification head trained fully |
| `bias` | none | Only LoRA matrices are trainable |

### Parameter Efficiency

- **Total parameters:** ~125M (RoBERTa-base)
- **Trainable parameters:** ~0.6M (LoRA matrices + classifier head)
- **Percentage trained:** <1%
- **Adapter size on disk:** ~3 MB (vs ~500 MB for full model)

## 4. Model Performance: With vs. Without LoRA Adapter

This is the core comparison. The base `roberta-base` model was never trained for
intent classification — it's a general-purpose language model. The LoRA adapter
transforms it into a specialized classifier.

In [ ]:
# Load evaluation results
with_adapter = pd.read_csv("../results/with_adapter/predictions.csv")
base_model = pd.read_csv("../results/base_model/predictions.csv")

# Overall metrics
metrics = {
    "Metric": ["Accuracy", "Correct predictions", "Misclassifications"],
    "With LoRA Adapter": [
        f"{(with_adapter['correct'].mean()):.1%}",
        f"{with_adapter['correct'].sum()} / {len(with_adapter)}",
        f"{(~with_adapter['correct']).sum()}",
    ],
    "Base Model (no adapter)": [
        f"{(base_model['correct'].mean()):.1%}",
        f"{base_model['correct'].sum()} / {len(base_model)}",
        f"{(~base_model['correct']).sum()}",
    ],
}

print("=" * 65)
print("OVERALL PERFORMANCE COMPARISON")
print("=" * 65)
pd.DataFrame(metrics).set_index("Metric")

In [ ]:
from sklearn.metrics import f1_score, classification_report

labels = sorted(id2label.keys())
label_names = [id2label[i] for i in labels]

# Compute per-class F1 for both
f1_adapter = f1_score(with_adapter["label"], with_adapter["predicted_label"],
                      labels=labels, average=None)
f1_base = f1_score(base_model["label"], base_model["predicted_label"],
                   labels=labels, average=None)

# Side-by-side bar chart
x = np.arange(len(label_names))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, f1_adapter, width, label="With LoRA Adapter", color="steelblue")
bars2 = ax.bar(x + width/2, f1_base, width, label="Base Model (no adapter)", color="lightcoral")

ax.set_ylabel("F1 Score")
ax.set_title("Per-Class F1 Score: LoRA Adapter vs. Base Model")
ax.set_xticks(x)
ax.set_xticklabels(label_names, rotation=45, ha="right")
ax.set_ylim(0, 1.15)
ax.legend()

for bar, score in zip(bars1, f1_adapter):
    if score > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{score:.2f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nLoRA adapter macro F1:  {np.mean(f1_adapter):.4f}")
print(f"Base model macro F1:    {np.mean(f1_base):.4f}")
print(f"Improvement:            {np.mean(f1_adapter) - np.mean(f1_base):.4f}")

## 5. Confusion Analysis

The confusion matrix reveals which intents the model confuses with each other.
Understanding these patterns helps identify where additional training data or
feature engineering could improve performance.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(with_adapter["label"], with_adapter["predicted_label"])
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names, ax=axes[0])
axes[0].set_title("Confusion Matrix (Counts)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
axes[0].tick_params(axis="x", rotation=45)

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[1].set_title("Confusion Matrix (Normalized)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Find top confusion pairs
print("\nTop confusion pairs (off-diagonal):")
for i in range(len(label_names)):
    for j in range(len(label_names)):
        if i != j and cm[i][j] > 0:
            print(f"  {label_names[i]:30s} → {label_names[j]:30s}  ({cm[i][j]} examples)")

## 6. Error Analysis

Let's examine the specific examples the model got wrong to understand the nature
of its mistakes — are they ambiguous cases, or clear failures?

In [ ]:
errors = with_adapter[~with_adapter["correct"]].copy()
errors["true_intent"] = errors["label"].map(id2label)

print(f"Total misclassifications: {len(errors)} out of {len(with_adapter)} ({len(errors)/len(with_adapter):.1%})\n")
print("=" * 90)

for _, row in errors.iterrows():
    print(f"  Text: \"{row['text']}\"")
    print(f"    True:      {row['true_intent']}")
    print(f"    Predicted: {row['predicted_intent']}")
    print(f"    Confidence: {row['confidence']:.3f}")
    print()

# Error distribution by true intent
print("\nErrors by true intent:")
error_counts = errors["true_intent"].value_counts()
for intent, count in error_counts.items():
    total = len(with_adapter[with_adapter["intent_name"] == intent])
    print(f"  {intent:30s}  {count}/{total} misclassified ({count/total:.1%})")

## 7. Confidence Distribution

A well-calibrated model should be more confident on correct predictions and less
confident on mistakes. Let's check if that holds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence for correct vs incorrect
correct = with_adapter[with_adapter["correct"]]["confidence"]
incorrect = with_adapter[~with_adapter["correct"]]["confidence"]

axes[0].hist(correct, bins=30, alpha=0.7, label=f"Correct (n={len(correct)})", color="steelblue")
axes[0].hist(incorrect, bins=30, alpha=0.7, label=f"Incorrect (n={len(incorrect)})", color="coral")
axes[0].set_xlabel("Confidence")
axes[0].set_ylabel("Count")
axes[0].set_title("Confidence Distribution: Correct vs. Incorrect")
axes[0].legend()
axes[0].axvline(x=0.5, color="gray", linestyle="--", alpha=0.5)

# Per-class average confidence
class_conf = with_adapter.groupby("intent_name")["confidence"].mean().sort_values()
class_conf.plot.barh(ax=axes[1], color="teal")
axes[1].set_xlabel("Mean Confidence")
axes[1].set_title("Average Prediction Confidence by Intent")

plt.tight_layout()
plt.show()

print(f"Mean confidence (correct):   {correct.mean():.4f}")
print(f"Mean confidence (incorrect): {incorrect.mean():.4f}")
print(f"Confidence gap:              {correct.mean() - incorrect.mean():.4f}")

## 8. Per-Class Performance Deep Dive

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    with_adapter["label"], with_adapter["predicted_label"],
    labels=labels, average=None
)

perf_df = pd.DataFrame({
    "Intent": label_names,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "Support": support.astype(int),
    "Class Weight": [class_weights[str(i)] for i in labels],
}).sort_values("F1 Score", ascending=True)

# Highlight weak spots
print("Per-class metrics (sorted by F1, ascending):\n")
print(perf_df.to_string(index=False, float_format="%.4f"))

print("\n\nIntents below 0.90 F1 (areas for improvement):")
weak = perf_df[perf_df["F1 Score"] < 0.90]
if len(weak) > 0:
    for _, row in weak.iterrows():
        print(f"  {row['Intent']:30s}  F1={row['F1 Score']:.3f}  "
              f"(P={row['Precision']:.3f}, R={row['Recall']:.3f}, n={row['Support']})")
else:
    print("  All intents above 0.90 F1!")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for i, name in enumerate(label_names):
    ax.scatter(recall[i], precision[i], s=support[i]*3, alpha=0.7, zorder=5)
    ax.annotate(name, (recall[i], precision[i]),
                textcoords="offset points", xytext=(8, 5), fontsize=8)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision vs. Recall by Intent (bubble size = test support)")
ax.set_xlim(0.55, 1.05)
ax.set_ylim(0.55, 1.05)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.axhline(y=0.9, color="gray", linestyle=":", alpha=0.5)
ax.axvline(x=0.9, color="gray", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

## 9. Conclusions & Production Considerations

### Key Findings

1. **LoRA is highly effective**: With <1% of parameters trained, we achieve 93% accuracy
   and 0.928 macro F1 — comparable to full fine-tuning performance.

2. **The adapter does all the work**: The bare base model scores ~3% accuracy (random chance
   for 10 classes = 10%), confirming that `roberta-base` has no innate understanding of
   maritime intents without the adapter.

3. **Class weights help rare intents**: Despite `report_port_incident` having only 8 test
   examples, the model achieves 100% recall on it (though precision suffers due to
   false positives from other classes).

4. **Main confusion areas**: `declare_cargo_manifest` ↔ `ask_vessel_schedule` and
   `request_pilotage_tug` show the most confusion, likely due to overlapping vocabulary
   (vessel names, dates, port terminology).

### Production Deployment

- **Storage**: Only the ~3MB LoRA adapter is stored per fine-tuned variant
- **Inference**: Load `roberta-base` once from HuggingFace cache, attach adapter via `PeftModel`
- **Multi-adapter serving**: Multiple domain adapters can share the same base model in memory
- **Monitoring**: Track confidence scores — low-confidence predictions should be flagged for human review

### Potential Improvements

- Add more training examples for confused intent pairs
- Experiment with higher LoRA rank (r=16 or r=32) for more capacity
- Try targeting all attention projections (Q, K, V, O) instead of just Q and V
- Implement confidence thresholding to route uncertain predictions to human agents